# LoRA v6 — HighRes512 Training

- Same proven v2 config (r=16, q/k/v/o, 1.1M params, 2 epochs, LR=3e-5)
- Only change: images resized to 512×512 instead of 224×224
- At 224, small text in diagrams, chart labels, and fine visual details get destroyed — the model literally can't see what it needs to answer
- Native resolution improved inference from 0.758 → 0.768 without retraining, so training at high-res should teach the model to use those details
- 512×512 gives 5.2x more pixels than 224×224
- Training will be slower per step (larger images = more compute) but same number of steps

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.0 MB/s eta 0:00:00


In [2]:
import kagglehub
from pathlib import Path
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

100%|██████████| 358M/358M [00:21<00:00, 17.1MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


In [4]:
import os, json, random, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

SEED = 42; random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"

BATCH_SIZE_TRAIN = 1
GRAD_ACCUM_STEPS = 16
LR = 3e-5
EPOCHS = 2
WARMUP_RATIO = 0.05

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

BATCH_SIZE_INFER = 4
SAVE_DIR = Path("/content/lora_v6")

from google.colab import drive
drive.mount("/content/drive")
DRIVE_SAVE = Path("/content/drive/MyDrive/lora_v6_highres512")

print(f"GPU: {torch.cuda.is_available()}")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists(): DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def build_prompt(row, include_answer=False):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    if include_answer:
        ans = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[ans]}. {choices[ans]}"
    return prompt

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    img = Image.open(path).convert("RGB")
    # Resize to fixed 512x512 (not thumbnail) so all images are same size
    img = img.resize((512, 512), Image.BICUBIC)
    return img

In [10]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (limit: 5,000,000)")
assert trainable <= 5_000_000

Trainable: 4,161,536 (limit: 5,000,000)


In [11]:
class VQATrainDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row)
        full_text = build_prompt(row, include_answer=True)
        full_inputs = processor(text=[full_text], images=[image], return_tensors="pt", truncation=False)
        answer_str = f" {CHOICE_LETTERS[int(row['answer'])]}. {row['choices'][int(row['answer'])]}"
        n_answer = len(processor.tokenizer.encode(answer_str, add_special_tokens=False))
        return {
            "input_ids": full_inputs["input_ids"].squeeze(0),
            "attention_mask": full_inputs["attention_mask"].squeeze(0),
            "pixel_values": full_inputs["pixel_values"].squeeze(0),
            "n_answer_tokens": n_answer,
        }

def collate_train(batch):
    max_len = max(x["input_ids"].shape[0] for x in batch)
    pad_id = processor.tokenizer.pad_token_id
    input_ids_list, attention_mask_list, labels_list, pixel_values_list = [], [], [], []
    for x in batch:
        seq_len = x["input_ids"].shape[0]
        pad_len = max_len - seq_len
        n_answer = x["n_answer_tokens"]
        input_ids = torch.cat([torch.full((pad_len,), pad_id, dtype=x["input_ids"].dtype), x["input_ids"]])
        attention_mask = torch.cat([torch.zeros(pad_len, dtype=x["attention_mask"].dtype), x["attention_mask"]])
        labels = torch.full_like(input_ids, -100)
        labels[max_len - n_answer:max_len] = input_ids[max_len - n_answer:max_len]
        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)
        pixel_values_list.append(x["pixel_values"])
    return {
        "input_ids": torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
        "labels": torch.stack(labels_list),
        "pixel_values": torch.stack(pixel_values_list),
    }

# Verify
print("Testing collator...")
_ds = VQATrainDataset(train_df.head(4))
_b = collate_train([_ds[j] for j in range(4)])
for i in range(4):
    n = (_b["labels"][i] != -100).sum().item()
    tok = _b["labels"][i][_b["labels"][i] != -100]
    print(f"  Sample {i}: {n} loss tokens, answer='{processor.tokenizer.decode(tok)[:60]}'")

Testing collator...
  Sample 0: 11 loss tokens, answer=' C. the male's tadpoles will become adult frogs'
  Sample 1: 9 loss tokens, answer=' A. the female's offspring will live longer'
  Sample 2: 10 loss tokens, answer=' B. the lioness's cubs will survive attacks'
  Sample 3: 8 loss tokens, answer=' B. the gull's offspring will survive'


In [12]:
subset = train_df.sample(frac=0.10, random_state=SEED).reset_index(drop=True)
ds_check = VQATrainDataset(subset)
loader_check = DataLoader(ds_check, batch_size=BATCH_SIZE_TRAIN, shuffle=True, collate_fn=collate_train, num_workers=0)

model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
losses = []; optimizer.zero_grad()

for step, batch in enumerate(loader_check):
    if step >= 200: break
    batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
    out = model(**batch)
    loss = out.loss / GRAD_ACCUM_STEPS
    loss.backward()
    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
    losses.append(out.loss.item())
    if (step+1) % 32 == 0:
        print(f"Step {step+1}: loss={losses[-1]:.4f} avg32={np.mean(losses[-32:]):.4f}")

print(f"\nFirst 32 avg: {np.mean(losses[:32]):.4f}")
print(f"Last 32 avg:  {np.mean(losses[-32:]):.4f}")

model.eval(); model.config.use_cache = True
del optimizer, loader_check, ds_check
gc.collect(); torch.cuda.empty_cache()

Step 32: loss=0.3966 avg32=0.6910
Step 64: loss=0.5121 avg32=0.5076
Step 96: loss=0.6463 avg32=0.5630
Step 128: loss=0.0782 avg32=0.4870
Step 160: loss=0.5516 avg32=0.3638
Step 192: loss=0.0987 avg32=0.3463

First 32 avg: 0.6910
Last 32 avg:  0.3470


In [13]:
del model; gc.collect(); torch.cuda.empty_cache()

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = get_peft_model(model, lora_config)
model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

ds_full = VQATrainDataset(train_df)
loader_full = DataLoader(ds_full, batch_size=BATCH_SIZE_TRAIN, shuffle=True,
                          collate_fn=collate_train, num_workers=2, pin_memory=True)

total_steps = len(loader_full) * EPOCHS
opt_steps = total_steps // GRAD_ACCUM_STEPS

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=max(1, int(opt_steps * WARMUP_RATIO)), num_training_steps=opt_steps,
)

loss_history = []; best_avg_loss = float("inf"); optimizer.zero_grad(); global_step = 0
CHECKPOINT_EVERY = 500

for epoch in range(EPOCHS):
    pbar = tqdm(loader_full, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(pbar):
        batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss / GRAD_ACCUM_STEPS
        loss.backward()
        if (global_step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
        loss_history.append(out.loss.item()); global_step += 1
        if global_step % 100 == 0:
            pbar.set_postfix(loss=f"{loss_history[-1]:.4f}", avg100=f"{np.mean(loss_history[-100:]):.4f}",
                             lr=f"{scheduler.get_last_lr()[0]:.2e}")
        if global_step % CHECKPOINT_EVERY == 0:
            current_avg = np.mean(loss_history[-CHECKPOINT_EVERY:])
            if current_avg < best_avg_loss:
                best_avg_loss = current_avg
                print(f"\n  💾 New best: {best_avg_loss:.4f} at step {global_step}")
                SAVE_DIR.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(SAVE_DIR); processor.save_pretrained(SAVE_DIR)
                import shutil
                DRIVE_SAVE.mkdir(parents=True, exist_ok=True)
                if DRIVE_SAVE.exists(): shutil.rmtree(DRIVE_SAVE)
                shutil.copytree(SAVE_DIR, DRIVE_SAVE)

model.eval(); model.config.use_cache = True
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR); processor.save_pretrained(SAVE_DIR)
import shutil
if DRIVE_SAVE.exists(): shutil.rmtree(DRIVE_SAVE)
shutil.copytree(SAVE_DIR, DRIVE_SAVE)
torch.cuda.empty_cache()
print(f"\nDone! First 50 avg: {np.mean(loss_history[:50]):.4f}, Last 50 avg: {np.mean(loss_history[-50:]):.4f}")

Epoch 1/2:   0%|          | 0/3109 [00:00<?, ?it/s]


  💾 New best: 0.4264 at step 500

  💾 New best: 0.1375 at step 1000

  💾 New best: 0.0937 at step 1500

  💾 New best: 0.0910 at step 2000

  💾 New best: 0.0765 at step 2500


Epoch 2/2:   0%|          | 0/3109 [00:00<?, ?it/s]


  💾 New best: 0.0676 at step 3500

  💾 New best: 0.0668 at step 4000

  💾 New best: 0.0611 at step 4500

  💾 New best: 0.0546 at step 6000

Done! First 50 avg: 0.5606, Last 50 avg: 0.0622


In [14]:
def get_candidate_token_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " "+letter, "\n"+letter, letter+".", " "+letter+"."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

candidate_ids = get_candidate_token_ids(processor.tokenizer)

def predict_batch(df_batch):
    images = [load_image(row) for _, row in df_batch.iterrows()]
    prompts = [build_prompt(row) for _, row in df_batch.iterrows()]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]
    log_probs = torch.log_softmax(logits, dim=-1)
    preds = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = [log_probs[i, candidate_ids[CHOICE_LETTERS[ci]]].max().item()
                  for ci in range(len(row["choices"]))]
        preds.append(int(np.argmax(scores)))
    return preds

In [15]:
model.eval(); gc.collect(); torch.cuda.empty_cache()
eval_df = val_df.copy().reset_index(drop=True)
print(f"Full validation: {len(eval_df)} samples")

all_preds = []
for start in tqdm(range(0, len(eval_df), BATCH_SIZE_INFER), desc="Val"):
    batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
    try:
        p = predict_batch(batch)
    except RuntimeError:
        torch.cuda.empty_cache()
        p = []
        for j in range(len(batch)):
            p.extend(predict_batch(batch.iloc[j:j+1]))
            torch.cuda.empty_cache()
    all_preds.extend(p)
    torch.cuda.empty_cache()

eval_df["pred"] = all_preds
acc = (eval_df["pred"] == eval_df["answer"]).mean()
print(f"\nFull val accuracy: {acc:.4f} ({(eval_df['pred'] == eval_df['answer']).sum()}/{len(eval_df)})")
print(f"Previous best leaderboard: 0.768")

Full validation: 1048 samples


Val:   0%|          | 0/262 [00:00<?, ?it/s]


Full val accuracy: 0.7405 (776/1048)
Previous best leaderboard: 0.768


In [16]:
all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="Test"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    try:
        p = predict_batch(batch)
    except RuntimeError:
        torch.cuda.empty_cache()
        p = []
        for j in range(len(batch)):
            p.extend(predict_batch(batch.iloc[j:j+1]))
            torch.cuda.empty_cache()
    all_preds.extend(p)
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_v6_highres.csv", index=False)
print(f"Saved ({len(submission)} rows)")

from google.colab import files
files.download("/content/submission.csv")

Test:   0%|          | 0/252 [00:00<?, ?it/s]

Saved (1008 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Hard Samples

In [ ]:
# Load v2 checkpoint, then run inference on TRAINING set
# to find which samples the model gets wrong

model.eval()
processor.tokenizer.padding_side = "left"

train_preds = []
for start in tqdm(range(0, len(train_df), BATCH_SIZE_INFER), desc="Scoring train set"):
    batch = train_df.iloc[start:start+BATCH_SIZE_INFER]
    p = predict_batch(batch)  # using native images + letter scoring
    train_preds.extend(p)
    torch.cuda.empty_cache()

train_df["pred"] = train_preds
train_df["correct"] = train_df["pred"] == train_df["answer"]

total = len(train_df)
correct = train_df["correct"].sum()
wrong = total - correct
print(f"Train accuracy: {correct}/{total} = {correct/total:.4f}")
print(f"Hard samples (wrong): {wrong}")

# Save hard samples
hard_df = train_df[~train_df["correct"]].copy()
hard_df.to_csv("/content/hard_samples.csv", index=False)
print(f"\nSaved {len(hard_df)} hard samples")

# Analyze what makes them hard
print(f"\n--- Hard vs Easy ---")
easy_df = train_df[train_df["correct"]]
print(f"Avg prompt length - Easy: {easy_df.apply(lambda r: len(build_prompt(r)), axis=1).mean():.0f} chars")
print(f"Avg prompt length - Hard: {hard_df.apply(lambda r: len(build_prompt(r)), axis=1).mean():.0f} chars")
print(f"Has lecture - Easy: {easy_df['lecture'].notna().mean():.2%}")
print(f"Has lecture - Hard: {hard_df['lecture'].notna().mean():.2%}")
print(f"Avg choices - Easy: {easy_df['choices'].apply(len).mean():.1f}")
print(f"Avg choices - Hard: {hard_df['choices'].apply(len).mean():.1f}")